In [1]:
import pandas as pd
import warnings
import numpy as np
import os
import sys
from pathlib import Path
from scipy.stats import norm
warnings.filterwarnings("ignore", category=RuntimeWarning) 

REPO = Path(__file__).resolve().parents[1] if '__file__' in dir() else Path(os.getcwd()).resolve().parent
os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))

from pkg import detrend_group

# Use yields.csv for year-level data, but filter to the same country-crop
# pairs in yield_comparison_df.csv (the YC model sample, N=1744) so that
# this analysis is on the same sample as the main model.
yc_pairs = pd.read_csv("./data/yield_comparison_df.csv")[["iso_a3", "cropname"]].drop_duplicates()

combined = pd.read_csv("./data/yields.csv")[["cropname", "year", "yield_og", "csif_og", "country", "iso_a3"]]
combined = combined.merge(yc_pairs, on=["iso_a3", "cropname"], how="inner")

combined = detrend_group(combined, 'yield_og', 'yield_log_dt', log_transform=True)
combined = detrend_group(combined, 'csif_og', 'csif_log_dt', log_transform=True)

# Create lag/lead of the unshifted detrended yield
combined['yield_lag'] = combined.groupby(['cropname', 'country'])['yield_log_dt'].shift(1)
combined['yield_lead'] = combined.groupby(['cropname', 'country'])['yield_log_dt'].shift(-1)

combined

,cropname,year,yield_og,csif_og,country,iso_a3,yield_log_dt,csif_log_dt,yield_lag,yield_lead
0,Barley,2000.0,596.8,0.030731,Afghanistan,AFG,-0.654474,-0.101442,NaN,-0.152609
1,Barley,2001.0,1000.0,0.026081,Afghanistan,AFG,-0.152609,-0.266662,-0.654474,0.212819
2,Barley,2002.0,1461.9,0.043133,Afghanistan,AFG,0.212819,0.235231,-0.152609,-0.298996
3,Barley,2003.0,888.9,0.039029,Afghanistan,AFG,-0.298996,0.134083,0.212819,-0.278264
4,Barley,2004.0,920.6,0.030863,Afghanistan,AFG,-0.278264,-0.101822,-0.298996,0.129625
...,...,...,...,...,...,...,...,...,...,...
40137,Yams,2019.0,10875.2,0.283696,Venezuela (Bolivarian Republic of),VEN,0.104347,-0.062164,-0.075512,-0.021184
40138,Yams,2020.0,9657.0,0.260898,Venezuela (Bolivarian Republic of),VEN,-0.021184,-0.149115,0.104347,0.002480
40139,Yams,2021.0,9955.0,0.334304,Venezuela (Bolivarian Republic of),VEN,0.002480,0.095630,-0.021184,0.011608
40140,Yams,2022.0,10114.1,0.313464,Venezuela (Bolivarian Republic of),VEN,0.011608,0.028084,0.002480,-0.000464


In [2]:
corrs = (combined.groupby(['country', 'cropname'])
    .apply(lambda g: g[['yield_log_dt', 'yield_lag', 'yield_lead']].corrwith(g['csif_log_dt'])))
corrs['argmax'] = corrs[['yield_log_dt', 'yield_lag', 'yield_lead']].idxmax(axis=1)
corrs['n'] = combined.groupby(['country', 'cropname'])['yield_log_dt'].count()
corrs = corrs.dropna()
corrs = corrs[corrs['n']>3]

In [3]:
df = corrs[corrs['argmax']!="yield_log_dt"].copy()

# Fisher's Z transformation
def fisher_z(r):
    return 0.5 * np.log((1 + r) / (1 - r))

# p-value calculation for comparing two correlations
def compare_correlations(r1, r2, n):
    z1 = fisher_z(r1)
    z2 = fisher_z(r2)
    diff = z1 - z2
    standard_error = np.sqrt(2 / (n - 3))  # Standard error for the difference
    z_stat = diff / standard_error
    p_value = 2 * norm.sf(abs(z_stat))  # Two-tailed p-value
    return p_value

# Apply the test
df['p_value'] = df.apply(
    lambda row: compare_correlations(
        row['yield_log_dt'],        # unshifted correlation
        row.at[row['argmax']],      # best lag/lead correlation  
        row['n']),
    axis=1)

In [4]:
print('total obs where lead/lags are better:', len(df),
     '\n', '\n',
     'obs where lead/lags are significantly better at p < 0.1:', len(df[df['p_value']<0.1]), 
     '\n', '\n',
     'percentage:', len(df[df['p_value'] < 0.1 ]) / len(df) * 100,
     '\n', '\n',
     'avg p-value:', df['p_value'].mean() )

total obs where lead/lags are better: 955 
 
 obs where lead/lags are significantly better at p < 0.1: 73 
 
 percentage: 7.643979057591623 
 
 avg p-value: 0.5222040750868441


In [5]:
supp_table = df[df['p_value']<0.1]
supp_table.columns=  ['corr y', 'corr y-1', 'corr y+1', 'argmax', 'n', 'p_value']
supp_table.to_csv("./plots/lag_sig_update.csv")

### Summary

Both statistics below are computed on the same sample used in the YC model
(`yield_comparison_df.csv`, N=1744 country–crop pairs) so the numbers are
directly comparable to the main results.  The full `yields.csv` has additional
pairs that lack one or more covariates (GDP, climate, etc.) and would inflate
the denominator.

In [6]:
total_pairs = len(corrs)
lead_lag_pairs = (corrs['argmax'] != 'yield_log_dt').sum()
sig_pairs = len(df[df['p_value'] < 0.1])

print(f"Total country-crop pairs (YC model sample): {total_pairs}")
print(f"Pairs assigned a lead/lag: {lead_lag_pairs} ({lead_lag_pairs/total_pairs*100:.1f}%)")
print(f"Of those, significantly better than unshifted (p<0.1, Fisher Z): {sig_pairs} ({sig_pairs/lead_lag_pairs*100:.1f}%)")

Total country-crop pairs (YC model sample): 1743
Pairs assigned a lead/lag: 955 (54.8%)
Of those, significantly better than unshifted (p<0.1, Fisher Z): 73 (7.6%)
